In [2]:
test_df_points_JOMR_nov25_BSD_02 = pd.read_csv('test_df_points_JOMR_nov25_BSD_02.csv')
indexed_df_points_JOMR_nov25_BSD_02 = point_indexeer(test_df_points_JOMR_nov25_BSD_02)
dict_bsd2 = score_checker(indexed_df_points_JOMR_nov25_BSD_02)
dict_bsd2

Scores au moment des switchs : [np.int64(5), np.int64(10), np.int64(15), np.int64(5), np.int64(10), np.int64(15), np.int64(20)]


{'équipes': ('JOMR', 'KaE_EmD'),
 'match_format': '15 points par set',
 'victoire': 'JOMR',
 'final_score': '2 - 0',
 'score_by_set': [{'set': 1, 'score': '14 - 5'},
  {'set': 2, 'score': '15 - 7'}]}

In [4]:
indexed_df_points_JOMR_nov25_BSD_01 = pd.read_csv('test_df_points_JOMR_nov25_BSD_01.csv')
dict_bsd1 = score_checker(indexed_df_points_JOMR_nov25_BSD_01)
dict_bsd1

Scores au moment des switchs : [np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(5), np.int64(10), np.int64(15), np.int64(20)]


{'équipes': ('JOMR', 'adversaire'),
 'match_format': '15 points par set',
 'victoire': 'JOMR',
 'final_score': '2 - 0',
 'score_by_set': [{'set': 1, 'score': '15 - 9'},
  {'set': 2, 'score': '15 - 7'}]}

In [10]:
indexed_df_points_JOMR_nov24 = pd.read_csv('indexed_df_points_JOMR_nov24_Leuven_01.csv')
dict_leuven = score_checker(indexed_df_points_JOMR_nov24)
dict_leuven

Scores au moment des switchs : [np.int64(7), np.int64(14), np.int64(21), np.int64(28), np.int64(7), np.int64(14), np.int64(21), np.int64(28), np.int64(35)]


{'équipes': ('JOMR', '(adv)'),
 'match_format': '21 points par set',
 'victoire': '(adv)',
 'final_score': '0 - 2',
 'score_by_set': [{'set': 1, 'score': '17 - 21'},
  {'set': 2, 'score': '15 - 21'}]}

In [2]:
from montage_auto_frames_gpu import point_indexeer
import pandas as pd
import os

In [13]:
test_df_points_JOMR_nov25_BSD_02 = pd.read_csv('test_df_points_JOMR_nov25_BSD_02.csv')
indexed_df_points_JOMR_nov25_BSD_02 = point_indexeer(test_df_points_JOMR_nov25_BSD_02)
indexed_df_points_JOMR_nov25_BSD_02.head()
indexed_df_points_JOMR_nov25_BSD_02.to_csv(
    path_or_buf=r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\indexed_df_points_JOMR_nov25_BSD_02.csv', 
    index=False)


In [6]:
import os

In [10]:
test_path = os.path.join(os.path.abspath(os.getcwd()), 'indexed_df_points')
test_path

'c:\\Users\\habib\\Documents\\GitHub\\DataBeach\\indexed_df_points'

In [12]:
test_path = r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_jan26_MBV_01'
game_ID = os.path.splitext(os.path.basename(test_path))[0]
team1_name, team2_name = game_ID.split('_')[1], game_ID.split('_')[2]
team1_name, team2_name

('jan26', 'MBV')

In [ ]:
def pipeline_point_editor_gpu(
    video_path:str,
    team1_name:str,
    team2_name:str,
    output_dir:str,
    indexed_df_points_dir:str = os.path.join(os.path.abspath(os.getcwd()), 'indexed_df_points'),
    recap_dict_score_dir:str = os.path.join(os.path.abspath(os.getcwd()), 'recap_dict_score')
):
    
    # [DEBUG]Check du path des dossiers de sortie
    print(f"Path du dossier 'indexed_df_points': {indexed_df_points_dir}")
    print(f"Path du dossier 'recap_dict_score': {recap_dict_score_dir}")

    # Vérification des répertoires 'indexed_df_points' et 'recap_dict_score'
    if not os.path.exists(indexed_df_points_dir) and not os.path.exists(recap_dict_score_dir):
        os.makedirs(indexed_df_points_dir)
        os.makedirs(recap_dict_score_dir)

    # Lecture de la vidéo et extraction des start_frame et end_frame
    df_points = pd.DataFrame()  # Initialisation d'un DataFrame vide
    df_points = cv2_point_segment_cut(
        video_path=video_path,
        team1_name=team1_name,
        team2_name=team2_name
        )
    # Indexation des points
    indexed_df_points = pd.DataFrame()  # Initialisation d'un DataFrame vide
    indexed_df_points = point_indexeer(df_points)
    # Score checking
    recap_dict_score = dict()  # Initialisation d'un dictionnaire vide
    recap_dict_score = score_checker(indexed_df_points)

    # Sauvegarde de l'indexation et du score dans des fichiers CSV et JSON
    indexed_df_points.to_csv(
        path_or_buf=os.path.join(indexed_df_points_dir, f'indexed_df_points_{game_ID}.csv'),
        index=False)
    with open(os.path.join(recap_dict_score_dir, f'recap_dict_score_{game_ID}.json'), 'w') as json_file:
        json.dump(recap_dict_score, json_file, indent=4)

    # Extraction des segments
    extract_segments_from_df_gpu(
        video_path=video_path,
        actions_df=indexed_df_points,
        output_dir=output_dir
    )

In [6]:
dir_path =os.path.join(os.path.abspath(os.getcwd()), 'points_segmented')
dir_path

'c:\\Users\\habib\\Documents\\GitHub\\DataBeach\\points_segmented'

In [6]:
import os
from montage_auto_frames_gpu import pipeline_point_editor_gpu

In [8]:
nom_video = 'JOMR_jan26_MBV_02.mp4'
video_path = os.path.join(os.path.abspath(os.getcwd()), 'matchs preprocess', nom_video)
video_path

'c:\\Users\\habib\\Documents\\GitHub\\DataBeach\\matchs preprocess\\JOMR_jan26_MBV_02.mp4'

In [13]:
video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_jan26_MBV_02.mp4'
video_path

'C:\\Users\\habib\\Desktop\\Montages volley et beach\\Jade&Math\\matchs preprocess\\JOMR_jan26_MBV_02.mp4'

In [ ]:
nom_video = 'JOMR_jan26_MBV_03.mp4'
video_path = os.path.join(r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess', nom_video)

pipeline_point_editor_gpu(
    video_path=video_path,
    team1_name='JOMR',
    team2_name='PauG_NeCH',
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented',
    indexed_df_points_dir=r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points',
    recap_dict_score_dir=r'C:\Users\habib\Documents\GitHub\DataBeach\recap_dict_score'
)

In [1]:
import cv2
import pandas as pd
import sys

def cv2_point_segment_cut(
    video_path : str,
    play_speed : float = 1.0,
    team1_name: str = "JOMR",
    team2_name: str = "adversaire"
    ) -> pd.DataFrame:

    """
    Création d'un dataframe contenant les segments de points extraits d'une vidéo de match, avec les informations sur le score et les équipes.
    A utiliser ensuite avec cut_point_gpu pour découper les segments de points à partir de ce dataframe. 
    Permet également de contrôler la lecture de la vidéo (pause, vitesse) pour faciliter l'identification des points.
    
    Args:
        video_path (str): Chemin vers la vidéo à traiter.
        play_speed (float, optional): Vitesse de lecture de la vidéo (1=normale, 0=pause, >1=plus rapide). Par défaut à 1.0.
        team1_name (str, optional): Nom de l'équipe 1. Par défaut à "JOMR".
        team2_name (str, optional): Nom de l'équipe 2. Par défaut à "adversaire".
        output_dir (str, optional): Dossier de sortie pour les segments extraits. Si None, les segments seront extraits dans le même dossier que la vidéo source.
    Returns:
        DataFrame contenant les informations sur les segments de points extraits, avec les colonnes : 
        'point_index',
        'action',
        'start_frame',
        'end_frame',
        'score_team1',
        'score_team2',
        'set_team1',
        'set_team2'
    """
    # Initialisation des variables
    temp_list = list()
    last_action = None

    # Mapping des touches aux temp_list
    key_action_map = {
        ord('0'): 'debut du set',
        ord('1'): f'service {team1_name}',
        ord('3'): f'service {team2_name}',
        ord('2'): 'fin point',
        ord('5'): '*SWITCH*',
        ord('8'): 'Temps mort',
        }
    
    # Afficher les touches disponibles en overlay sur la vidéo
    help_lines = [
        "0 : debut du set",
        f"1 : service {team1_name}",
        f"3 : service {team2_name}",
        "2 : fin du point",
        "5 : switch",
        "8 : temps mort"
    ]

    # Initialiser les scores pour affichage en overlay
    score_team1 = 0
    score_team2 = 0

    # Redéfinir cv2.imshow pour ajouter l'overlay d'aide et les scores
    _orig_imshow = cv2.imshow

    def _imshow_with_help(winname, frame):
        if frame is not None:
            # Afficher l'aide en haut à gauche
            x, y = 30, 120
            for i, line in enumerate(help_lines):
                cv2.putText(frame,
                            line,
                            (x, y + i * 15),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.3,
                            (255, 255, 255),
                            1,
                            cv2.LINE_AA)
            
            # Afficher les scores en bas à droite
            h, w = frame.shape[:2]
            score_text = f"{team1_name}: {score_team1}  {team2_name}: {score_team2}"
            text_size = cv2.getTextSize(score_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
            score_x = w - text_size[0] - 20
            score_y = h - 20
            cv2.putText(frame,
                        score_text,
                        (score_x, score_y),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2,
                        cv2.LINE_AA)
        _orig_imshow(winname, frame)

    cv2.imshow = _imshow_with_help


    # Ouvrir la vidéo
    cap = cv2.VideoCapture(video_path)

    def _waitKey_fast(ms):
        # Réduire le délai proportionnellement à la vitesse (au moins 1 ms)
        adj = max(1, int(ms / play_speed))
        return cv2.waitKey(adj)

    if not cap.isOpened():
        print("Erreur : impossible d’ouvrir la vidéo.")
        sys.exit()

    # Récupérer les FPS pour convertir les frames en temps
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_number = 0

    # État de pause et rotation
    paused = False

    # Boucle de lecture de la vidéo
    try:
        while cap.isOpened():

            # Lire une frame seulement si on n'est pas en pause
            if not paused:
                ret, frame = cap.read()

                # Arrêter si fin de vidéo ou erreur
                if not ret:
                    print("Fin de la vidéo ou erreur de lecture.")
                    break

                # Incrémenter le compteur de frames
                frame_number += 1

            # Affiche la dernière action
            if last_action and ret:
                cv2.putText(frame,
                            f"Derniere action : {last_action}",
                            (30, 40),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,
                            (0, 255, 0),
                            2,
                            cv2.LINE_AA)

            # Indiquer le mode pause sur l'image affichée
            if paused and ret:
                cv2.putText(frame,
                            "|| PAUSE ||",
                            (30, 80),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,
                            (0, 0, 255),
                            2,
                            cv2.LINE_AA)

            # Afficher la frame courante
            if ret:
                cv2.imshow(f'{video_path}', frame)

            # Gestion des entrées clavier
            key = _waitKey_fast(30) & 0xFF
            if key == ord('q'):  # quitter
                break
            elif key == ord(' '):  # pause/reprise
                paused = not paused
            elif key == ord('+'):  # augmenter la vitesse
                play_speed += 0.5
                continue
            elif key == ord('-'):  # diminuer la vitesse
                play_speed = max(0.5, play_speed - 0.5)
                continue
            elif key in key_action_map: # enregistrer l'action associée à la touche, avec le numéro de frame
                if key == ord('0') and len(temp_list) == 0: # début du set est en fait début du match
                    action_name = str('debut du match')
                    # reset des scores en début de set
                    score_team1 = 0
                    score_team2 = 0
                else:
                    action_name = key_action_map[key]
   
                last_action = action_name
                temp_list.append({
                    'Frame': frame_number,
                    'Action': action_name
                })
                
                # Refresh de l'affichage pour mettre à jour les scores en overlay
                if ret:
                    cv2.imshow(f'{video_path}', frame)
                # Ajout d'un point aux scores en fonction de l'action
                if action_name == f'service {team1_name}':
                    score_team1 += 1
                elif action_name == f'service {team2_name}':
                    score_team2 += 1

            elif key == ord('7'): # erreur de codage, revenir en arrière
                if temp_list:                  
                    removed_action = temp_list.pop()
                    print(f"Action supprimée : {removed_action}")
                    last_action = temp_list[-1]['Action'] if temp_list else None
                else:
                    print("Aucune action à supprimer.")
                # Refresh de l'affichage
                if ret:
                    cv2.imshow(f'{video_path}', frame)                  
                
                # Revenir à la frame de l'action supprimée et mettre la lecture en pause
                if temp_list:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, temp_list[-1]['Frame'])
                    frame_number = temp_list[-1]['Frame']
                    paused = True
                else:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    frame_number = 0
                    paused = True

    finally:
        # Libérer les ressources OpenCV
        cap.release()
        cv2.destroyAllWindows()


    # Initialiser une liste pour stocker les segments de points
    list_actions = list()

    # Traiter la temp_list pour créer les segments de points avec les informations de score et de set
    i = 0
    while i < len(temp_list):
        action = temp_list[i]
        
        # Si l'action n'est pas 'fin point', on la traite
        if action['Action'] != 'fin point':
            # Si c'est un début de set, création d'un item 'fin du set' dans la list_action pour traitement ultérieur
            if action['Action'] == 'debut du set':
                list_actions.append({
                    'Service_side': 'fin du set',
                    'Start_frame': int(0),
                    'End_frame': int(0)
                })
            # Puis on traite l'action normalement
            start_frame = action['Frame']
            service_side = action['Action']
            # Chercher le prochain 'fin point'
            end_frame = None
            j = i + 1
            while j < len(temp_list):
                if temp_list[j]['Action'] == 'fin point':
                    end_frame = temp_list[j]['Frame']
                    break
                j += 1
            
            # Ajouter à la liste si on a trouvé un 'fin point'
            if end_frame is not None:
                list_actions.append({
                    'Service_side': service_side,
                    'Start_frame': start_frame,
                    'End_frame': end_frame
                })

        i += 1

    # Convertir la list_actions en DataFrame 
    df_points = pd.DataFrame(list_actions)

    # Ajouter des colonnes pour les scores et les sets, initialisées à 0
    df_points[f'{team1_name}_score'] = 0
    df_points[f'{team2_name}_score'] = 0
    df_points[f'{team1_name}_sets'] = 0
    df_points[f'{team2_name}_sets'] = 0

    # Mettre à jour les scores en fonction du service_side
    for idx, row in df_points.iterrows():
        if df_points['Service_side'].iloc[idx] == 'debut du set' or df_points['Service_side'].iloc[idx] == 'debut du match':
            # Première ligne : initialiser selon le service
            if row['Service_side'] == f'service {team1_name}':
                df_points.at[idx, f'{team1_name}_score'] = 1
                df_points.at[idx, f'{team2_name}_score'] = 0
            elif row['Service_side'] == f'service {team2_name}':
                df_points.at[idx, f'{team1_name}_score'] = 0
                df_points.at[idx, f'{team2_name}_score'] = 1
        else:
            # Lignes suivantes : reprendre le score précédent et incrémenter selon le service
            df_points.at[idx, f'{team1_name}_score'] = df_points.at[idx - 1, f'{team1_name}_score']
            df_points.at[idx, f'{team2_name}_score'] = df_points.at[idx - 1, f'{team2_name}_score']
            
            if row['Service_side'] == f'service {team1_name}':
                df_points.at[idx, f'{team1_name}_score'] += 1
            elif row['Service_side'] == f'service {team2_name}':
                df_points.at[idx, f'{team2_name}_score'] += 1    

    # Mettre à jour les scores de sets au début de chaque nouveau set
    for idx, row in df_points.iterrows():
        if row['Service_side'] == 'debut du set' and idx > 0:
            # Comparer les scores du set précédent
            prev_score_team1 = df_points.at[idx - 1, f'{team1_name}_score']
            prev_score_team2 = df_points.at[idx - 1, f'{team2_name}_score']
            
            # Récupérer les sets précédents
            df_points.at[idx, f'{team1_name}_sets'] = df_points.at[idx - 1, f'{team1_name}_sets']
            df_points.at[idx, f'{team2_name}_sets'] = df_points.at[idx - 1, f'{team2_name}_sets']
            
            # Ajouter un set au gagnant
            if prev_score_team1 > prev_score_team2:
                df_points.at[idx, f'{team1_name}_sets'] += 1
            else:
                df_points.at[idx, f'{team2_name}_sets'] += 1
        elif idx > 0:
            # Pour les autres lignes, conserver le nombre de sets
            df_points.at[idx, f'{team1_name}_sets'] = df_points.at[idx - 1, f'{team1_name}_sets']
            df_points.at[idx, f'{team2_name}_sets'] = df_points.at[idx - 1, f'{team2_name}_sets']
  

    # Ajout d'une ligne de 'fin de set' à la fin de df_points pour faciliter le traitement ultérieur
    df_points.loc[len(df_points)] = {
        'Service_side': 'fin du set',
        'Start_frame': int(0),
        'End_frame': int(0),
        f'{team1_name}_score': df_points.at[len(df_points) - 1, f'{team1_name}_score'],
        f'{team2_name}_score': df_points.at[len(df_points) - 1, f'{team2_name}_score'],
        f'{team1_name}_sets': df_points.at[len(df_points) - 1, f'{team1_name}_sets'],
        f'{team2_name}_sets': df_points.at[len(df_points) - 1, f'{team2_name}_sets']
    }

    # Pour les lignes de 'fin du set', mettre à jour les scores de sets en fonction du score du set précédent
    for idx, row in df_points.iterrows():
        if row['Service_side'] == 'fin du set' and idx > 0:
            # Le gagnant du point précédent et du set précédent est celui qui avait le score le plus élevé au point précédent
            prev_score_team1 = df_points.at[idx - 1, f'{team1_name}_score']
            prev_score_team2 = df_points.at[idx - 1, f'{team2_name}_score']
            df_points.at[idx, f'{team1_name}_sets'] = df_points.at[idx - 1, f'{team1_name}_sets']
            df_points.at[idx, f'{team2_name}_sets'] = df_points.at[idx - 1, f'{team2_name}_sets']
            if prev_score_team1 > prev_score_team2:
                df_points.at[idx, f'{team1_name}_sets'] += 1
                df_points.at[idx, f'{team1_name}_score'] += 1
            elif prev_score_team2 > prev_score_team1:
                df_points.at[idx, f'{team2_name}_sets'] += 1
                df_points.at[idx, f'{team2_name}_score'] += 1
            

    return df_points

In [2]:
test_df_points = cv2_point_segment_cut(
    video_path = r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_jan26_MBV_03.mp4',
    play_speed = 1.0,
    team1_name = "JOMR",
    team2_name = "XXXX"
    )

KeyError: -1